In [8]:
import os
# os.chdir('../')

In [9]:
import pandas as pd
import pickle

from textwave.modules.retrieval.index.bruteforce import FaissBruteForce
from textwave.modules.retrieval.search import FaissSearch

from textwave.modules.generator.question_answering import QAGeneratorMistral
from textwave.modules.retrieval.reranker import Reranker
from textwave.modules.utils.metrics import Matching

print('DONE')

DONE


In [21]:
QUESTIONS_PATH = 'textwave/qa_resources/question.tsv'
CORPUS_PATH = 'textwave/storage/'
CHUNKING_STRATEGY = 'fixed-length' # 'fixed-length' or 'sentence'
CHUNKING_PARAMETERS = {
    "chunk_size": 150, 
    "overlap_size": 0
}
INDEX_STRATEGY = "bruteforce"
INDEX_PARAMETERS = {
    'metric': 'cosine',
}
K_NEAREST_NEIGHBORS = 3
MISTRAL_MODEL = 'mistral-large-latest'
# mistral-small-latest
# mistral-medium-latest
# mistral-large-latest
API_KEY = os.environ["MISTRAL_API_KEY"]

In [22]:
# Process questions df
raw_questions = pd.read_table(QUESTIONS_PATH)
easy = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'easy'].reset_index(drop=True)[:20]
medium = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'medium'].reset_index(drop=True)[:20]
hard = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'hard'].reset_index(drop=True)[:20]
questions = pd.concat([easy, medium, hard]).reset_index(drop=True)

In [15]:
# Use best performing index
index = FaissBruteForce.load('faiss/bruteforce_cosine_fixed-length_150_0.pkl')

In [16]:
# Load question embeddings
with open('analysis/question_embeddings.pkl', 'rb') as f:
    embeddings_map = pickle.load(f)

In [ ]:
from sentence_transformers import CrossEncoder
model = CrossEncoder(
    "zli12321/answer_equivalence_distilbert",
    num_labels=2
)

# from sentence_transformers import CrossEncoder
# model = CrossEncoder(
#     "cross-encoder/ms-marco-TinyBERT-L-2-v2",
#     # num_labels=2
# )

In [23]:
# Initialize objects
mistral = QAGeneratorMistral(API_KEY, generator_model=MISTRAL_MODEL)
search = FaissSearch(index, metric=INDEX_PARAMETERS['metric'])
rerank = Reranker(type='tfidf')

# Get unique questions 
unique_questions = questions['Question'].unique()
results = {}
count = 0
for question in unique_questions:
    print(f'{count+1}/{len(unique_questions)}')

    # Embed query
    query_vector = embeddings_map[question]

    # Search index for neighbors
    _, _, meta_results = search.search(query_vector, k=K_NEAREST_NEIGHBORS)

    # Perform re-ranking - choose "tfidf re-rank" for this
    reranked_context, _, _ = rerank.rerank(question, meta_results, seq_k1=5, seq_k2=3) # End up with 3 context chunks to match previous attempts

    # Trigger QA object to ping MISTRAL, get reponse, return
    answer = mistral.generate_answer(query=question, context=reranked_context)
    results[question] = answer

    # Increment count
    count += 1

results

1/41
2/41
3/41
4/41
5/41
6/41
7/41
8/41
9/41
10/41
11/41
12/41
13/41
14/41
15/41
16/41
17/41
18/41
19/41
20/41
21/41
22/41
23/41
24/41
25/41
26/41
27/41
28/41
29/41
30/41
31/41
32/41
33/41
34/41
35/41
36/41
37/41
38/41
39/41
40/41
41/41


{'Was Abraham Lincoln the sixteenth President of the United States?': 'Yes, Abraham Lincoln was the sixteenth President of the United States. He served from March 4, 1861, until his death on April 15, 1865, and was elected on November 6, 1860.',
 'Did Lincoln sign the National Banking Act of 1863?': 'No context.',
 'Did his mother die of pneumonia?': 'No, his mother did not die of pneumonia. She died of **milk sickness** at the age of thirty-four.',
 "How many long was Lincoln's formal education?": "Lincoln's formal education lasted about **18 months**.",
 'When did Lincoln begin his political career?': 'Abraham Lincoln began his political career in **1832**, at the age of **23**.',
 'What did The Legal Tender Act of 1862 establish?': 'The Legal Tender Act of 1862 established the **United States Note**, the first paper currency in the U.S., also known as the **greenback currency**, which was issued during the Civil War and pledged to be redeemable in gold.',
 'Was Abraham Lincoln the f

In [24]:
with open('results.pkl', 'wb') as file:
    pickle.dump(results, file)

In [25]:
# Load pickle file
with open('results.pkl', 'rb') as file:
    results = pickle.load(file)

# Get metrcis, add to dataframe
metrics = Matching(model='cross-encoder/nli-distilroberta-base')

processed = questions[~questions['Question'].isna()]
processed = processed[~processed['Answer'].isna()]
indices = []
for idx, row in processed.iterrows():
    if row['Question'] not in results:
        indices.append(idx)
processed = processed.drop(indices)

for idx, row in processed.iterrows():
    question = row['Question']
    print(f'{idx+1}/{len(processed)+1}')

    true_answer = row['Answer']
    generated_answer = results[question]

    em = metrics.exact_match(generated_answer, true_answer)
    print(f"Exact Match: {em}")

    # try:
    #     scores, match = metrics.transformer_match(generated_answer, true_answer, question)
    #     print(f"Transformer Match: {match} | Scores: {scores}\n")
    # except:
    #     match = None
    #     print('Failed to get transformer match!')

    processed.at[idx, 'Exact Match'] = em
    # processed.at[idx, 'Transformer Match'] = match

1/60
Exact Match: True
2/60
Exact Match: True
3/60
Exact Match: False
4/60
Exact Match: True
5/60
Exact Match: True
6/60
Exact Match: True
7/60
Exact Match: False
8/60
Exact Match: True
9/60
Exact Match: True
10/60
Exact Match: True
11/60
Exact Match: False
12/60
Exact Match: False
13/60
Exact Match: True
14/60
Exact Match: True
15/60
Exact Match: True
16/60
Exact Match: False
17/60
Exact Match: False
18/60
Exact Match: True
19/60
Exact Match: False
20/60
Exact Match: False
21/60
Exact Match: False
22/60
Exact Match: True
23/60
Exact Match: True
24/60
Exact Match: True
25/60
Exact Match: False
26/60
Exact Match: False
27/60
Exact Match: False
28/60
Exact Match: True
29/60
Exact Match: True
30/60
Exact Match: False
31/60
Exact Match: False
32/60
Exact Match: True
33/60
Exact Match: False
34/60
Exact Match: False
35/60
Exact Match: False
36/60
Exact Match: True
37/60
Exact Match: False
38/60
Exact Match: True
39/60
Exact Match: False
40/60
Exact Match: True
41/60
Exact Match: False
42/60

In [26]:
n = processed[~processed['Exact Match'].isna()]
easy = n[n['DifficultyFromQuestioner'] == 'easy']
medium = n[n['DifficultyFromQuestioner'] == 'medium']
hard = n[n['DifficultyFromQuestioner'] == 'hard']

dfs = [easy, medium, hard]

for df in dfs:
    print(len(df[df['Exact Match']==True]) / len(df))
    # print(len(df[df['Transformer Match']==True]) / len(df))

0.75
0.625
0.25
